# 10 - Synthetic Front-View Augmentation Plan

Planning only, no images generated here. Synthetic images can only ever be used as training-time augmentation, never for validation/test, and generation doesn't start until this plan is approved. Keeping the plan and prompt templates here next to the code rather than only in a separate doc.

## Why synthetic data might be needed at all

Some camera views (e.g. `overhead`, `mirror`) may end up poorly represented or missing entirely from the real datasets. If a view/class combination still has too few real examples after AUC V2/100-Driver, synthetic images could fill that gap during training only — never as the actual measure of whether the model works.

## Hard rules for this track

1. Synthetic images = training augmentation only, never test/validation data.
2. Real data used for validation/testing wherever possible.
3. No image generation until this plan is explicitly approved, separately from the rest of the multi-view stage.
4. Every synthetic image tagged `is_synthetic=True` in the unified schema (see `README.md` / `data/README.md`), so it can be filtered out of evaluation and results reported with/without it.
5. Left/right still follows the driver's-own-left/right rule — a generated front-camera image of the driver texting with their real right hand is labelled `texting_right`, matching the mirrored-appearance rule for real front-view datasets.

## Prompt templates (recorded now, not used yet)

Stored as plain data below so they're version-controlled and reviewable — not run against any image generation API here.

In [1]:
# NOTE: this cell only defines text templates - it does not call any image generation service.

PROMPT_TEMPLATE = (
    "Photorealistic in-car driver monitoring image, adult driver, {view}, "
    "visible face upper body hands, steering wheel and dashboard visible, {action}, "
    "realistic lighting, no text, no watermark, no logo."
)

NEGATIVE_PROMPT = (
    "cartoon, anime, CGI, painting, blurry, distorted hands, extra fingers, missing fingers, "
    "wrong steering wheel, outside car, child, crash, accident, blood, text, watermark, logo, celebrity"
)

VIEWS = [
    "front camera mounted near dashboard facing driver",
    "passenger-side camera view",
    "driver-side camera view",
    "rear-view mirror camera angle",
    "overhead cabin camera view",
]

ACTIONS = {
    "safe_driving": "safe driving, both hands on steering wheel",
    "texting_right": "texting with right hand, phone near lap",
    "phone_right": "talking on phone at right ear with right hand",
    "texting_left": "texting with left hand, phone near lap",
    "phone_left": "talking on phone at left ear with left hand",
    "adjusting_radio": "operating radio or infotainment screen",
    "drinking": "drinking from bottle or can",
    "reaching_behind": "reaching behind seat with one arm",
    "hair_or_makeup": "fixing hair or makeup using mirror",
    "talking_to_passenger": "talking to passenger",
}

print(f"{len(VIEWS)} views x {len(ACTIONS)} actions = {len(VIEWS) * len(ACTIONS)} possible prompt combinations (not generated).")

5 views x 10 actions = 50 possible prompt combinations (not generated).


## Gap analysis

SAM-DD is merged now (notebook 08) so this table is real, just still missing AUC V2 and 100-Driver. SAM-DD's `front` view has zero examples of `adjusting_radio` and `talking_to_passenger` — likely because the simulator rig's front camera didn't capture those actions. That's the kind of specific gap the approval checklist below needs before generating anything, but not something to act on yet since AUC V2/100-Driver might fill it with real data.

In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
unified_path = PROJECT_ROOT / "data" / "processed" / "unified_multiview_metadata.csv"
unified_df = pd.read_csv(unified_path)

coverage = unified_df.groupby(["camera_view", "label_name"]).size().unstack(fill_value=0)
display(coverage)

datasets_included = sorted(unified_df["dataset_name"].unique())
print(f"This table currently reflects: {datasets_included} (camera views: {sorted(unified_df['camera_view'].unique())}).")
print("Re-run after AUC V2 / 100-Driver are merged to see whether any (view, class) gaps remain once those are added.")

label_name,adjusting_radio,drinking,hair_or_makeup,phone_left,phone_right,reaching_behind,safe_driving,talking_to_passenger,texting_left,texting_right
camera_view,,,,,,,,,,
front,0,3459,2061,3048,2931,1771,28264,0,3163,3074
side,2312,5784,3972,5374,5248,3773,30753,2129,5509,5341


This table currently reflects: ['sam_dd', 'state_farm'] (camera views: ['front', 'side']).
Re-run after AUC V2 / 100-Driver are merged to see whether any (view, class) gaps remain once those are added.


## Approval checklist (before any image is generated)

- [ ] Real multi-view data (AUC V2 and/or 100-Driver) merged via notebook 08
- [ ] Gap-analysis table above shows a specific, real (view, class) shortage — not a guess
- [ ] Explicit approval given for generating synthetic images for that specific gap
- [ ] Image generation tool/service chosen (not decided yet)
- [ ] Generated images saved to a separate folder (e.g. `data/synthetic/`) and tagged `is_synthetic=True`, `dataset_name='synthetic'`
- [ ] Evaluation always reported both with and without synthetic-augmented training, never mixed into test data

Status: none of the above done yet — plan only.